# Tres redes y sus errores

**Sesión 1 · IA Intermedio · EstudIA 2026**

Vamos a usar tres redes neuronales que otra persona entrenó y a ponerlas a prueba:

| Parte | Red | Datos de entrenamiento | Tarea |
|---|---|---|---|
| 1 | CNN (ResNet-50) | ImageNet: fotos etiquetadas en 1.000 clases | Decir qué hay en la foto |
| 2 | Transformer (BERT multilingüe) | Reseñas de productos en seis idiomas | Calificar una reseña de 1 a 5 estrellas |
| 3 | MLP (Keras) | Clientes de un banco portugués (2008-2010) | Predecir si un cliente acepta un depósito a término |

Con cada red veremos ejemplos en los que acierta y otros en los que falla. Tu trabajo es explicar
**por qué** pasa cada cosa. Para eso, compara cada ejemplo con los datos con los que se entrenó la red.
Las partes 2 y 3 terminan con un ejercicio corto de código.

**Antes de empezar**
1. *Archivo › Guardar una copia en Drive*, para trabajar en tu propia copia.
2. No necesitas GPU: el entorno gratuito de Colab basta.
3. Ejecuta la celda de preparación (botón ▶ o `Shift + Enter`) y espera el mensaje “Listo”. Descarga unos 1,2 GB de modelos; en Colab tarda uno o dos minutos.

In [ ]:
# Preparación: descarga modelos, imágenes y datos. Ejecútala una vez.
ARCHIVOS = "https://storage.googleapis.com/estud-ia-course-estudia-lab/s01"

import io, json, os, urllib.request, warnings
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from IPython.display import display, Markdown
from transformers import pipeline
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

def descargar(ruta):
    """Trae un archivo del curso (fotos, datos y el modelo del banco)."""
    req = urllib.request.Request(f"{ARCHIVOS}/{ruta}", headers={"User-Agent": "EstudIA-notebook/1.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

CREDITOS = {
    "tucan.jpg": "Bernard DUPONT, CC BY-SA 2.0, Wikimedia Commons",
    "colibri.jpg": "Elio Rafael Ariza Ramos, CC BY-SA 4.0, Wikimedia Commons",
    "arepas.jpg": "Instituto de Turismo del Meta (Constantino Castelblanco), CC BY-SA 4.0, Wikimedia Commons",
    "bandeja-paisa.jpg": "TitiNicola, CC BY-SA 4.0, Wikimedia Commons",
    "metrocable.jpg": "Bernard Gagnon, CC BY-SA 4.0, Wikimedia Commons",
    "esqueleto.jpg": "Museum of Veterinary Anatomy FMVZ USP, CC BY-SA 4.0, Wikimedia Commons",
    "gato.jpg": "Alvesgaspar, CC BY-SA 3.0, Wikimedia Commons",
}
IMAGENES = {n: Image.open(io.BytesIO(descargar(f"img/{n}"))).convert("RGB") for n in CREDITOS}

print("Cargando la CNN (ResNet-50)…")
cnn = pipeline("image-classification", model="microsoft/resnet-50", device=-1)
print("Cargando los modelos de texto…")
resenas = pipeline("text-classification", model="nlptown/bert-base-multilingual-uncased-sentiment", device=-1, top_k=None)
tuits = pipeline("text-classification", model="pysentimiento/robertuito-sentiment-analysis", device=-1, top_k=None)

print("Cargando el MLP del banco…")
import keras
open("bank_mlp.keras", "wb").write(descargar("bank_mlp.keras"))
PREP = json.loads(descargar("bank_prep.json"))
BANCO = pd.read_csv(io.BytesIO(descargar("bank.csv.gz")), compression="gzip")

def preparar(df):
    """Estandariza columnas numéricas y convierte categorías en columnas 0/1, igual que en el entrenamiento."""
    partes = [((df[c].astype(float) - PREP["mean"][c]) / PREP["std"][c]).to_numpy()[:, None] for c in PREP["numeric"]]
    partes += [np.stack([(df[c] == v).to_numpy(dtype=float) for v in PREP["vocab"][c]], axis=1) for c in PREP["categorical"]]
    return np.concatenate(partes, axis=1).astype("float32")

try:
    mlp = keras.saving.load_model("bank_mlp.keras", compile=False)
except Exception as e:  # versión de Keras distinta: entrena la misma red aquí (unos 30 s)
    print("No pude cargar el modelo guardado; lo entreno aquí con la misma arquitectura.", e)
    tr = BANCO[BANCO.split == "train"]
    keras.utils.set_random_seed(42)
    mlp = keras.Sequential([keras.Input((preparar(tr[:1]).shape[1],)), keras.layers.Dense(32, activation="relu"),
                            keras.layers.Dense(16, activation="relu"), keras.layers.Dense(1, activation="sigmoid")])
    mlp.compile("adam", "binary_crossentropy")
    mlp.fit(preparar(tr), (tr.y == "yes").to_numpy("float32"), epochs=15, batch_size=128, verbose=0)

def prob_si(df):
    """Probabilidad de que cada cliente acepte el depósito."""
    return mlp.predict(preparar(df), verbose=0).ravel()

# ---------- ayudas para mostrar resultados
def ver(nombre, k=3, titulo=None):
    img = IMAGENES[nombre]
    preds = cnn(img, top_k=k)
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [1.1, 1]})
    a.imshow(img); a.axis("off"); a.set_title(titulo or nombre, fontsize=11)
    etiquetas = [p["label"].split(",")[0] for p in preds][::-1]
    b.barh(etiquetas, [p["score"] for p in preds][::-1], color="#2b2b2b")
    b.set_xlim(0, 1); b.set_xlabel("probabilidad"); b.spines[["top", "right"]].set_visible(False)
    for i, p in enumerate(preds[::-1]):
        b.text(min(p["score"] + 0.02, 0.86), i, f"{p['score']:.2f}", va="center")
    fig.text(0.01, -0.02, f"Foto: {CREDITOS[nombre]}", fontsize=8, color="#5b6170")
    plt.tight_layout(); plt.show()
    return preds

def buscar_clase(palabra):
    """Busca una palabra entre las 1.000 clases de ImageNet que conoce la CNN."""
    clases = cnn.model.config.id2label.values()
    hallazgos = [c for c in clases if palabra.lower() in c.lower()]
    print(f"“{palabra}”: {hallazgos if hallazgos else 'no está entre las 1.000 clases'}")

def donde_mira(nombre, cuadros=7):
    """Tapa la foto por partes y mide cuánto cae la probabilidad de la clase ganadora.
    Rojo intenso = zona que la red necesita para decidir."""
    img = IMAGENES[nombre].resize((224, 224))
    ganadora = cnn(img, top_k=1)[0]
    lado = 224 // cuadros
    tapadas = []
    for f in range(cuadros):
        for c in range(cuadros):
            x = img.copy(); ImageDraw.Draw(x).rectangle([c * lado, f * lado, (c + 1) * lado, (f + 1) * lado], fill=(128, 128, 128))
            tapadas.append(x)
    probs = [next((p["score"] for p in r if p["label"] == ganadora["label"]), 0.0) for r in cnn(tapadas, top_k=5)]
    caida = np.clip(ganadora["score"] - np.array(probs), 0, None).reshape(cuadros, cuadros)
    fig, ax = plt.subplots(1, 2, figsize=(9, 4))
    ax[0].imshow(img); ax[0].axis("off"); ax[0].set_title(f"{ganadora['label'].split(',')[0]} ({ganadora['score']:.2f})")
    ax[1].imshow(img); ax[1].imshow(np.kron(caida, np.ones((lado, lado))), cmap="Reds", alpha=0.6, vmin=0, vmax=max(caida.max(), 1e-6))
    ax[1].axis("off"); ax[1].set_title("Dónde mira")
    fig.text(0.01, -0.02, f"Foto: {CREDITOS[nombre]}", fontsize=8, color="#5b6170")
    plt.tight_layout(); plt.show()

def estrellas(textos):
    """Califica frases con el modelo de reseñas (1 a 5 estrellas)."""
    filas = []
    for t in textos:
        r = sorted(resenas(t)[0] if isinstance(resenas(t)[0], list) else resenas(t), key=lambda o: -o["score"])
        filas.append({"frase": t, "predicción": r[0]["label"], "confianza": round(r[0]["score"], 2),
                      "segunda opción": f"{r[1]['label']} ({r[1]['score']:.2f})"})
    return pd.DataFrame(filas)

def comparar(textos):
    """Compara el modelo de reseñas con uno entrenado en tuits en español."""
    def top(m, t):
        r = m(t); r = r[0] if isinstance(r[0], list) else r
        b = max(r, key=lambda o: o["score"]); return f"{b['label']} ({b['score']:.2f})"
    return pd.DataFrame([{"frase": t, "reseñas (1-5 estrellas)": top(resenas, t), "tuits (NEG/NEU/POS)": top(tuits, t)} for t in textos])

def cliente(fila_id, **cambios):
    """Muestra un cliente del conjunto de prueba, con cambios opcionales, y la predicción del MLP."""
    df = BANCO.loc[[fila_id]].copy()
    for k, v in cambios.items():
        df[k] = v
    p = prob_si(df)[0]
    cols = ["age", "job", "balance", "duration", "campaign", "poutcome", "y"]
    display(df[cols].rename(columns={"age": "edad", "job": "ocupación", "balance": "saldo (€)", "duration": "duración llamada (s)",
                                     "campaign": "llamadas en campaña", "poutcome": "campaña anterior", "y": "¿aceptó? (real)"}))
    print(f"Probabilidad de que acepte según el MLP: {p:.1%}  →  predicción: {'sí' if p >= 0.5 else 'no'}")
    return p

# ---------- revisiones de los ejercicios
def revisar_desacuerdo(frase):
    if frase is None:
        return print("Todavía no hay respuesta. Completa los ___ y ejecuta la celda del ejercicio; si ya lo hiciste, revisa la condición del if.")
    r = max(resenas([frase])[0], key=lambda o: o["score"])
    t = max(tuits([frase])[0], key=lambda o: o["score"])["label"]
    e = int(r["label"][0])
    if (e <= 2 and t == "POS") or (e >= 4 and t == "NEG"):
        print(f"¡Bien! El modelo de reseñas le da {e} estrellas y el de tuits responde {t}.")
    else:
        print(f"Esa no es. El modelo de reseñas le da {e} estrellas y el de tuits responde {t}: los dos están de acuerdo.")

def revisar_quiebre(segundos, paso=10):
    if segundos is None:
        return print("Todavía no hay respuesta. Completa los ___ y ejecuta la celda del ejercicio; si ya lo hiciste, revisa la condición del if.")
    fila = BANCO.loc[[17104]].copy()
    p = lambda s: prob_si(fila.assign(duration=s))[0]
    if p(segundos) >= 0.5 and (segundos < paso or p(segundos - paso) < 0.5):
        print(f"¡Bien! Con {segundos} s (unos {segundos / 60:.0f} minutos) el MLP pasa a “sí”: {p(segundos):.1%}.")
    elif p(segundos) < 0.5:
        print(f"Con {segundos} s el MLP todavía dice “no” ({p(segundos):.1%}). Busca más arriba.")
    else:
        print(f"Con {segundos} s el MLP ya dice “sí”, pero también con {segundos - paso} s. Busca la primera.")

print("Listo.")

---
## Parte 1. Una CNN que reconoce fotos

**ResNet-50** es una red convolucional que Microsoft entrenó con **ImageNet**: 1,28 millones de fotos
etiquetadas en **1.000 clases** (perros, aves, objetos, comidas…). Su última capa tiene 1.000 neuronas,
una por clase, y una función *softmax* que reparte el 100 % de la probabilidad entre ellas.

### 1.1 Un tucán

In [ ]:
ver("tucan.jpg");

**Para pensar.**

- ¿Por qué crees que la red acierta con tanta seguridad? Ejecuta `buscar_clase("toucan")` en una celda nueva.
- ¿Qué tan parecida crees que es esta foto a las fotos de tucanes con las que se entrenó?

### 1.2 Un colibrí colombiano

*Lepidopyga lilliae*, el colibrí de Lilly, vive solo en manglares del Caribe colombiano y está en peligro crítico.

In [ ]:
ver("colibri.jpg", titulo="Lepidopyga lilliae (colibrí)");
buscar_clase("hummingbird")
buscar_clase("jacamar")

**Para pensar.**

- ¿Qué responde la red y con qué probabilidad? ¿Qué tiene esta foto que la acerca a esa respuesta? (Busca fotos de jacamares).
- Mira la salida de `buscar_clase`. ¿La clase correcta está entre las 1.000? ¿Por qué crees que la red falla? Piensa en el tamaño del ave en la foto y en qué colibríes aparecen en ImageNet.
- ¿Qué te dice este caso sobre la confianza (la probabilidad) de una predicción?

**¿Dónde mira la red?** Tapamos la foto por partes y medimos cuánto cae la probabilidad. Las zonas rojas son las que la red necesita para decidir.

In [ ]:
donde_mira("tucan.jpg")
donde_mira("colibri.jpg")

**Para pensar.**

- En el tucán, ¿la red mira el ave o el fondo? ¿Y en el colibrí?
- Si la red mirara sobre todo el fondo y aun así acertara, ¿confiarías en ella con fotos nuevas?

### 1.3 Arepas

Una foto de arepas de arroz. Antes de ejecutar: ¿qué crees que va a responder la red?

In [ ]:
ver("arepas.jpg");
buscar_clase("arepa")

**Para pensar.**

- Mira la salida de `buscar_clase`. ¿Qué opciones tenía la red para responder?
- ¿Por qué crees que la probabilidad es tan alta?
- Si fueras a usar esta red en una app de comida colombiana, ¿qué cambiarías?

### 1.4 Más casos para discutir

In [ ]:
for n, t in [("bandeja-paisa.jpg", "Bandeja paisa"), ("metrocable.jpg", "Metrocable de Medellín"),
             ("esqueleto.jpg", "Esqueleto de un perro"), ("gato.jpg", "Un gato")]:
    ver(n, titulo=t)

**Para pensar.**

- Clasifica cada caso: ¿la clase correcta existe entre las 1.000? ¿La red escogió la clase “más cercana”?
- En el gato, la red duda entre tres clases de gato. ¿Una persona sabría cuál es la correcta? ¿El error es de la red o de la etiqueta?

**Prueba con tu propia foto** (opcional). Toma una foto de algo que tengas cerca.

In [ ]:
from google.colab import files
subida = files.upload()
for nombre, datos in subida.items():
    IMAGENES[nombre] = Image.open(io.BytesIO(datos)).convert("RGB")
    CREDITOS[nombre] = "tu foto"
    ver(nombre)

---
## Parte 2. Un transformer que califica reseñas

`nlptown/bert-base-multilingual-uncased-sentiment` es un BERT multilingüe ajustado con **reseñas de
productos** en seis idiomas, entre ellos el español. Recibe un texto y responde de **1 a 5 estrellas**.

### 2.1 Una reseña clara

In [ ]:
estrellas(["La atención fue pésima y el pedido llegó incompleto."])

**Para pensar.**

- ¿Qué palabras crees que pesaron en la decisión?

### 2.2 Una reseña con sarcasmo

In [ ]:
estrellas(["Excelente servicio: dos horas esperando y la pizza llegó fría."])

**Para pensar.**

- ¿Qué tanto se parece esta frase a los datos con los que se entrenó la red? ¿Por qué crees que falla?
- ¿Qué tendría que “entender” la red para acertar? ¿Una persona lo entiende siempre?

### 2.3 Parlache

In [ ]:
estrellas(["Ese parche estuvo una chimba, parce."])

**Para pensar.**

- Para alguien de Medellín la frase es muy positiva. ¿Por qué la red se equivoca?
- ¿Con qué datos tendrías que entrenar o ajustar la red para que entienda el parlache?

### 2.4 Más casos

In [ ]:
estrellas([
    "Pedí una hamburguesa.",
    "Nunca vuelvo a pedir aquí.",
    "Nunca había comido algo tan rico.",
    "El producto es excelente pero el envío fue un desastre.",
])

**Para pensar.**

- “Pedí una hamburguesa.” no expresa ninguna opinión. ¿Qué opción le falta a esta red?
- La red acierta con “Nunca vuelvo a pedir aquí.” y falla con “Nunca había comido algo tan rico.”. ¿Acertó la primera por la razón correcta? Prueba en una celda nueva tres frases positivas con “nunca”.
- ¿Cuántas estrellas merece la frase con “pero”? ¿Estarían de acuerdo dos personas?

### 2.5 Otro modelo, otros datos

`pysentimiento/robertuito-sentiment-analysis` es un RoBERTa entrenado con **tuits en español** y responde NEG, NEU o POS. Compáralo con el modelo de reseñas.

In [ ]:
comparar([
    "La atención fue pésima y el pedido llegó incompleto.",
    "Excelente servicio: dos horas esperando y la pizza llegó fría.",
    "Ese parche estuvo una chimba, parce.",
    "Pedí una hamburguesa.",
    "Nunca había comido algo tan rico.",
])

**Para pensar.**

- ¿Qué cambió entre los dos modelos? Mira con qué datos entrenaron cada uno.
- Si tu proyecto usara mensajes de clientes de Medellín, ¿cuál escogerías como punto de partida y qué harías después?

### Ejercicio 1. ¿En qué frase no están de acuerdo?

Los dos modelos usan escalas distintas: el de reseñas responde de 1 a 5 estrellas y el de tuits
responde NEG, NEU o POS. Decimos que **no están de acuerdo** cuando uno ve la frase negativa y el otro
positiva: **2 estrellas o menos con POS**, o **4 estrellas o más con NEG**.

En esta lista hay **una sola** frase así. Completa los `___` para encontrarla. Primero predice todas
las frases con cada modelo y después compara las respuestas frase por frase.

- `resenas(FRASES)` devuelve una lista con una entrada por frase. Cada entrada es una lista de
  etiquetas ordenada de la más probable a la menos probable, así:
  `[{"label": "5 stars", "score": 0.79}, {"label": "4 stars", "score": 0.14}, …]`
- `tuits(FRASES)` devuelve lo mismo con las etiquetas `"NEG"`, `"NEU"` y `"POS"`.
- `int("5 stars"[0])` convierte la etiqueta en el número 5.

**Intenta resolverlo por tu cuenta antes de mirar la solución.** Después ejecuta la celda de revisión.

In [ ]:
FRASES = [
    "El domicilio llegó rápido y la comida estaba deliciosa.",
    "Me dejaron metido una hora y nadie respondió.",
    "Ese restaurante es una nota, súper bacano.",
    "El producto llegó roto y no me devolvieron la plata.",
    "Genial, ahora la app ni siquiera abre.",
    "Qué boleta ese servicio, parce.",
    "Jamás había probado una bandeja tan buena.",
    "Muy caro para lo que ofrecen.",
]

respuesta_1 = None

pred_resenas = ___   # las predicciones de las 8 frases, en una sola llamada
pred_tuits = ___

for frase, r, t in zip(FRASES, pred_resenas, pred_tuits):
    estrellas = ___   # número de 1 a 5 de la etiqueta más probable
    tono = ___        # "NEG", "NEU" o "POS"
    if ___:
        respuesta_1 = frase
        print(f"{frase}  →  {estrellas} estrellas, {tono}")

In [ ]:
revisar_desacuerdo(respuesta_1)

In [ ]:
# @title Solución del ejercicio 1 (ábrela solo si te atascas)
pred_resenas = resenas(FRASES)
pred_tuits = tuits(FRASES)

respuesta_1 = None
for frase, r, t in zip(FRASES, pred_resenas, pred_tuits):
    estrellas = int(r[0]["label"][0])
    tono = t[0]["label"]
    if (estrellas <= 2 and tono == "POS") or (estrellas >= 4 and tono == "NEG"):
        respuesta_1 = frase
        print(f"{frase}  →  {estrellas} estrellas, {tono}")

**Para pensar.**

- ¿Cuál de los dos modelos tiene razón con esa frase? ¿Qué palabra crees que confundió al otro?
- Las demás frases tienen parlache, “jamás” o quejas sin palabras negativas fuertes, y aun así los dos modelos coinciden. ¿Coincidir significa que los dos acertaron?

**Prueba tus frases.**

In [ ]:
comparar(["Escribe aquí tu frase", "Y otra más"])

---
## Parte 3. Un MLP que predice clientes

Un banco portugués llamó por teléfono a 45.211 clientes entre 2008 y 2010 para ofrecerles un
depósito a término (parecido a un CDT). De cada cliente sabemos, entre otras cosas, su edad, su
ocupación (una lista fija de categorías), su saldo en euros, cuánto duró la llamada y cómo le fue en la
campaña anterior. Con esos datos entrenamos una red **MLP** en Keras: 51 entradas,
dos capas ocultas (32 y 16 neuronas) y una salida con la probabilidad de que el cliente acepte.
La entrenamos antes de clase con el 80 % de los clientes; en la sesión 2 veremos cómo.

Aquí la evaluamos con el otro 20 %, que no usamos para entrenar.

### 3.0 ¿Qué tan buena es?

In [ ]:
prueba = BANCO[BANCO.split == "test"].copy()
prueba["p"] = prob_si(prueba)
real = prueba.y == "yes"; pred = prueba.p >= 0.5
print(f"Clientes de prueba: {len(prueba)}  ·  aceptaron: {real.mean():.1%}")
print(f"Exactitud del MLP:                 {(pred == real).mean():.1%}")
print(f"Exactitud de responder “no” a todos: {(~real).mean():.1%}")
print(f"De los que aceptaron, el MLP encontró (sensibilidad):  {pred[real].mean():.1%}")
print(f"De los que el MLP marcó “sí”, aceptaron (precisión):   {real[pred].mean():.1%}")
display(pd.crosstab(real.map({True: "aceptó", False: "no aceptó"}), pred.map({True: "predice sí", False: "predice no"})))

**Para pensar.**

- La exactitud del MLP parece alta. ¿Contra qué deberías compararla?
- Si cada llamada cuesta y cada cliente que acepta deja ganancia, ¿qué error le cuesta más al banco?

### 3.1 Un acierto

In [ ]:
cliente(44521);

**Para pensar.**

- Mira la columna “campaña anterior”. ¿Por qué crees que el MLP está tan seguro?

### 3.2 Un error

In [ ]:
cliente(42326);

**Para pensar.**

- Este cliente aceptó y el MLP le dio casi 0 %. ¿Qué información le faltaría al modelo para acertar?

### 3.3 El mismo cliente, en pesos

Un banco de Medellín quiere reutilizar el modelo con sus clientes. Su sistema guarda el saldo en pesos
colombianos (1 € ≈ 4.300 COP). Pasamos el cliente del 3.1 con el saldo en pesos.

In [ ]:
cliente(44521, balance=BANCO.loc[44521, "balance"] * 4300);

aceptaron = prueba[real & pred].copy()
en_pesos = aceptaron.assign(balance=aceptaron.balance * 4300)
print(f"Clientes bien clasificados como “sí” que pasan a “no” en pesos: {(prob_si(en_pesos) < 0.5).sum()} de {len(aceptaron)}")

**Para pensar.**

- Es la misma persona. ¿Qué cambió para la red?
- El saldo más alto del entrenamiento fue 102.127 €. Compáralo con el saldo en pesos de este cliente. ¿Qué crees que hace la red con ese número?
- ¿Cómo detectarías este problema antes de usar el modelo con clientes reales?

### 3.4 Acertar por la razón equivocada

El cliente 17104 aceptó, y el MLP acierta. La llamada duró 1.448 segundos (24 minutos).
¿Qué pasa si la llamada dura 90 segundos?

In [ ]:
cliente(17104);
cliente(17104, duration=90);

### Ejercicio 2. El punto de quiebre

Con 1.448 segundos el MLP dice “sí”; con 90 segundos dice “no”. ¿A partir de cuántos segundos de
llamada cambia a “sí”? Completa los `___`: prueba duraciones de 10 en 10 segundos y detente en la
primera con probabilidad de 0,5 o más.

- `fila` es una tabla de una sola fila con el cliente 17104. Cambia su columna `"duration"`.
- `prob_si(fila)` devuelve un arreglo con una probabilidad por fila; `[0]` toma la primera.

**Intenta resolverlo por tu cuenta antes de mirar la solución.** Después ejecuta la celda de revisión.

In [ ]:
fila = BANCO.loc[[17104]].copy()

respuesta_2 = None
for segundos in range(0, 1501, 10):
    fila["duration"] = ___
    p = prob_si(___)[0]
    if ___:
        respuesta_2 = segundos
        print(f"Con {segundos} s la probabilidad es {p:.1%}: el MLP cambia a “sí”.")
        break

In [ ]:
revisar_quiebre(respuesta_2)

In [ ]:
# @title Solución del ejercicio 2 (ábrela solo si te atascas)
fila = BANCO.loc[[17104]].copy()

respuesta_2 = None
for segundos in range(0, 1501, 10):
    fila["duration"] = segundos
    p = prob_si(fila)[0]
    if p >= 0.5:
        respuesta_2 = segundos
        print(f"Con {segundos} s la probabilidad es {p:.1%}: el MLP cambia a “sí”.")
        break

**Para pensar.**

- ¿El banco conoce la duración de la llamada **antes** de llamar?
- Si el modelo depende de un dato que solo existe después del resultado, ¿sirve para decidir a quién llamar? A esto se le llama **fuga de información** (*data leakage*).

### 3.5 Una ocupación nueva

In [ ]:
print("Ocupaciones del entrenamiento:", PREP["vocab"]["job"])
cliente(44521, job="influencer");

**Para pensar.**

- Compara “influencer” con las ocupaciones del entrenamiento. ¿Qué hizo el modelo con ella? ¿Avisó algo?
- ¿Qué debería hacer un sistema en producción en un caso así?

---
## Cierre

Antes de la discusión, intenta agrupar los casos que viste: ¿cuáles se parecen a los datos de
entrenamiento y cuáles no? ¿En cuáles la red acertó por una razón que no te convence? ¿En cuáles ni
siquiera una persona estaría segura de la respuesta?

Lleva tus ideas a la discusión: con estos casos vamos a hablar de **análisis de errores** y de
**qué error le cuesta más a quien usa el modelo**.

### Créditos de las fotos

Todas las fotos vienen de Wikimedia Commons y se usan según su licencia.

| Foto | Autor | Licencia |
|---|---|---|
| [Tucán](https://commons.wikimedia.org/wiki/File:Toco_Toucan_(Ramphastos_toco)_-_48153967707.jpg) | Bernard DUPONT | CC BY-SA 2.0 |
| [Colibrí de Lilly](https://commons.wikimedia.org/wiki/File:Lepidopyga_lilliae.png) | Elio Rafael Ariza Ramos | CC BY-SA 4.0 |
| [Arepas de arroz](https://commons.wikimedia.org/wiki/File:Arepas_de_arroz.jpg) | Instituto de Turismo del Meta (Constantino Castelblanco) | CC BY-SA 4.0 |
| [Bandeja paisa](https://commons.wikimedia.org/wiki/File:Bandeja_Paisa_-_Comida_tipica_colombiana.jpg) | TitiNicola | CC BY-SA 4.0 |
| [Metrocable, línea K](https://commons.wikimedia.org/wiki/File:L%C3%ADnea_K_(Medell%C3%ADn_Metrocable)_03.jpg) | Bernard Gagnon | CC BY-SA 4.0 |
| [Esqueleto de perro](https://commons.wikimedia.org/wiki/File:Golden_Retriever_dog_at_MAV-USP.jpg) | Museum of Veterinary Anatomy FMVZ USP | CC BY-SA 4.0 |
| [Gato](https://commons.wikimedia.org/wiki/File:Cat_November_2010-1a.jpg) | Alvesgaspar | CC BY-SA 3.0 |